# 03 -- Three-Timescale User Model

**Author**: Tamer Atesyakar

The I3 user model maintains *three* running statistics over behavioural features:
- **Instant** -- last observation.
- **Session** -- short-horizon EMA (~ last few minutes).
- **Long-term** -- slow EMA over many sessions.

Their sufficient statistics are updated online using Welford's algorithm.

**Citations**
- Welford (1962). *Note on a method for calculating corrected sums of squares and products.* Technometrics.
- Knuth (1998). *The Art of Computer Programming*, vol. 2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
C_COLD = '#0f3460'
C_HOT  = '#e94560'
plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.25})

## 3.1 Welford's online variance

Naive two-pass variance is catastrophically unstable on streams. Welford's recurrence keeps the running mean `M` and the sum of squared deviations `S` and only needs `O(1)` memory per feature:

```
n   += 1
d    = x - M
M   += d / n
S   += d * (x - M)
var  = S / (n - 1)
```

We verify it matches `np.var(ddof=1)`.

In [ ]:
class Welford:
    def __init__(self):
        self.n = 0; self.M = 0.0; self.S = 0.0
    def push(self, x):
        self.n += 1
        d = x - self.M
        self.M += d / self.n
        self.S += d * (x - self.M)
    @property
    def var(self):
        return self.S / max(self.n - 1, 1)
    @property
    def std(self):
        return self.var ** 0.5

rng = np.random.default_rng(7)
xs = rng.normal(loc=3.0, scale=1.5, size=5000)
w = Welford()
for x in xs:
    w.push(x)
print(f'Welford mean={w.M:.6f}   numpy mean={xs.mean():.6f}')
print(f'Welford var ={w.var:.6f}  numpy var ={xs.var(ddof=1):.6f}')

## 3.2 Exponential moving averages at three timescales

An EMA with smoothing `alpha` has effective memory `~ 1/alpha` observations. We pick three alphas corresponding to instant (alpha=1), session (alpha=0.12 -> ~8 messages) and long-term (alpha=0.01 -> ~100 messages).

In [ ]:
class EMA:
    def __init__(self, alpha, init=0.0):
        self.alpha = alpha
        self.v = float(init)
        self.started = False
    def push(self, x):
        if not self.started:
            self.v = float(x); self.started = True
        else:
            self.v = (1 - self.alpha) * self.v + self.alpha * float(x)
        return self.v

# Simulate 200 messages where the user transitions: relaxed -> stressed -> relaxed
T = 200
load = np.zeros(T, dtype=np.float32)
rng = np.random.default_rng(3)
for t in range(T):
    if   t <  70:  base = 0.3   # relaxed
    elif t < 130:  base = 0.75  # stressed burst
    else:          base = 0.4
    load[t] = np.clip(base + rng.normal(0, 0.1), 0, 1)

inst = EMA(alpha=1.0)
sess = EMA(alpha=0.12)
long_ = EMA(alpha=0.01)

traces = []
for x in load:
    traces.append([inst.push(x), sess.push(x), long_.push(x)])
traces = np.array(traces)
traces.shape

## 3.3 Visualising the three traces

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(load,          color='#999', lw=1.0, alpha=0.65, label='raw signal')
ax.plot(traces[:, 0],  color='#333', lw=0.7, alpha=0.6,  label='instant (alpha=1)')
ax.plot(traces[:, 1],  color=C_COLD, lw=2.0, label='session (alpha=0.12)')
ax.plot(traces[:, 2],  color=C_HOT,  lw=2.0, label='long-term (alpha=0.01)')
ax.axvspan(70, 130, color=C_HOT, alpha=0.07, label='stressed burst')
ax.set_xlabel('message index')
ax.set_ylabel('cognitive load')
ax.set_title('Three-timescale EMA traces over 200 messages')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()

## 3.4 Deviation as an anomaly signal

A powerful derived signal is the *deviation* between the session and long-term traces, standardised by the long-run variance. Large positive deviations flag acute stress above baseline.

In [ ]:
# Running Welford on the raw signal gives the long-run variance.
w = Welford()
zs = []
for t in range(T):
    w.push(load[t])
    sigma = max(w.std, 1e-3)
    zs.append((traces[t, 1] - traces[t, 2]) / sigma)
zs = np.array(zs)

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(zs, color=C_HOT, lw=1.4)
ax.axhline(0, color='#555', lw=0.5)
ax.axhline(1.5, color=C_COLD, lw=0.8, ls='--', label='+1.5 sigma')
ax.axhline(-1.5, color=C_COLD, lw=0.8, ls='--')
ax.fill_between(np.arange(T), zs, 0, where=(np.abs(zs) > 1.5),
                color=C_HOT, alpha=0.25)
ax.set_title('Session-minus-baseline deviation (standardised)')
ax.set_xlabel('message index'); ax.set_ylabel('z-score')
ax.legend()
plt.tight_layout(); plt.show()

## 3.5 Why three timescales?

- **Instant** preserves responsiveness: a single outlier can still trigger a safety intervention.
- **Session** captures the current mood trajectory.
- **Long-term** acts as an *anchor* so that the system does not drift with transient variation.

The online Welford update (1962) gives the long-term variance with constant memory and numerically stable deviation z-scores.